In [1]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00


**Built-in Tool - DuckDuckGo Search**

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke("tell me about the current update of today's whether")

print(results)

Want to know what the weather is now? Check out our current live radar and weather forecasts to help plan your day Latest weather forecast for United States for today's, hourly weather forecast, including today's temperatures in the United States, wind, rain and more. Weather Tomorrow Offers a 15 day long-range forecast or an hour by hour forecast for the current day. Data is available for major cities of the world. NOAA National Weather ServiceA 20 percent chance of showers and thunderstorms after 1pm. Mostly sunny, with a high near 94. Heat index values as high as 99. West wind around 5 mph becoming north in the afternoon. Get accurarate 10-day weather forecast for your area: today, tomorrow and beyond. Daily temperature highs, lows, precipitation chances, and wind speed to plan your week!


In [4]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


**Built-in Tool - Shell Tool**

In [5]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke("ls")

print(results)

Executing command:
 ls
sample_data



/usr/local/lib/python3.11/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


**Custom Tools**

In [6]:
from langchain_core.tools import tool

In [7]:
# Step 1 - create a function

def multiply(a,b):
  """Multiply two numbers"""
  return a*b

In [8]:
# Step 2 - add type hints

def multiply(a:int ,b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [9]:
# Step 3 - add tool decorator

@tool
def multiply(a:int , b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [10]:
result = multiply.invoke({"a":3, "b":5})

In [11]:
print(result)

15


In [12]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [13]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


**Method 2 - Using Structured Tool**

In [14]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

In [15]:
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="The first number to add")
  b: int = Field(required=True, description="The second number to add")

In [16]:
def multiply_func(a:int , b:int) -> int:
  return a * b

In [17]:
multiply_tool = StructuredTool.from_function(
    func= multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [18]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


**Method 3 - Using Base Tool Class**

In [19]:
from langchain.tools import BaseTool
from typing import Type

In [20]:
#arg schema using pydantic

class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="The first number to add")
  b: int = Field(required=True, description="The second number to add")

In [21]:
class MultiplyTool(BaseTool):
  name : str = "multiply"
  description: str = "Multiply two numbers"

  args_schema: Type[BaseModel] = MultiplyInput

  def _run(self, a:int , b:int) -> int:
    return a * b

In [22]:
mutiply_tool = MultiplyTool()

In [23]:
result = multiply_tool.invoke({"a":3, "b":3})

In [24]:
print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


**Tool Kit**

In [25]:
from langchain_core.tools import tool

@tool
def add(a: int,b:int) -> int:
  """Add two numbers"""
  return a + b

@tool
def multiply(a: int, b:int) -> int:
  """Multiply two numbers"""
  return a*b

In [26]:
class MathToolKit:
  def get_tools(self):
    return [add , multiply]

In [27]:
toolkit = MathToolKit()
tools = toolkit.get_tools()

for tool in tools:
  print(tool.name, "=>" , tool.description)

add => Add two numbers
multiply => Multiply two numbers
